# Fed-BioMed training a model on a CSV dataset harmonized with FedCombat

This example shows how to use a federated preprocessing for harmonization datasets with FedCombat method, before training a model on the harmonized dataset. The dataset used on each node is a CSV format file. The example CSV file is synthetic data with a format inspired from ADNI dataset.

This example uses Pseudo Adni Dataset. Please check `README.md` file in `notebooks` directory for the instructions to load Pseudo Adni dataset and configure nodes.

## Create an experiment to train a model on the data found

Declare a torch training plan MyTrainingPlan class to send for training on the node

In [ ]:
import torch
import torch.nn as nn
from fedbiomed.common.training_plans import TorchTrainingPlan
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.dataset import TabularDataset
import pandas as pd

# Here we define the model to be used. 
# You can use any class name (here 'MyTrainingPlan')
class MyTrainingPlan(TorchTrainingPlan):
    
    # Model 
    def init_model(self, model_args):    
        model = self.Net(model_args)
        return model 
    
    
    # Dependencies
    def init_dependencies(self):
        # Here we define the custom dependencies that will be needed by our custom Dataloader
        # In this case, we need the torch Dataset and DataLoader classes
        # We need pandas to read the local .csv file at the node side
        deps = ["from fedbiomed.common.dataset import TabularDataset"]
        
        return deps
    
    class Net(nn.Module):
        def __init__(self, model_args):
            super().__init__()
            # should match the model arguments dict passed below to the experiment class
            self.fc1 = nn.Linear(model_args['in_features'], 5)
            self.fc2 = nn.Linear(5, model_args['out_features'])

        def forward(self, x):
            x = self.fc1(x)
            x = F.relu(x)
            x = self.fc2(x)
            return x

    def training_step(self, data, target):
        output = self.model().forward(data).float()
        criterion = torch.nn.MSELoss()
        loss   = criterion(output, target)
        return loss

    def training_data(self):
        x_dim = self.model_args()['in_features']
        dataset = TabularDataset(
            input_columns=list(range(1,x_dim+1)),
            target_columns='APOE4',
            transform=lambda xs: torch.as_tensor(xs, dtype=torch.float32),
            target_transform=lambda xs: torch.as_tensor(xs, dtype=torch.float32)
        )

        train_kwargs = {'shuffle': True}
        
        return DataManager(dataset=dataset, **train_kwargs)

In [ ]:
# model parameters 
model_args = {
    'in_features': 15, 
    'out_features': 1
}

# training parameters 
training_args = {
    'loader_args': { 'batch_size': 20, }, 
    'optimizer_args': {
        'lr': 1e-3
    }, 
    'epochs': 2, 
    'dry_run': False,  
}

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage

# Calling the training data with specified tags
#tags =  ['adni-unbalanced']
tags = ['adni']
rounds = 2

exp = Experiment(tags=tags,
                 training_plan_class=MyTrainingPlan,
                 model_args=model_args,
                 training_args=training_args,
                 round_limit=rounds,
                 aggregator=FedAverage(),
                 node_selection_strategy=None,
                 save_breakpoints=True,
                 tensorboard=True
                )

In [ ]:
tensorboard_dir = exp.tensorboard_results_path
tensorboard_dir

In [ ]:
# Uncomment        fc_model_training for tensorboard 
#%load_ext tensorboard
#%tensorboard --logdir "$tensorboard_dir"

In [ ]:
#exp.training_plan_file()

### Set pre-processing

`Phenotypes` are the variables from the dataset that are harmonized by Fed-ComBat (the target features to correct for batch/site effects).

`Covariates` are the variables whose effects are modeled and preserved during harmonization (e.g., age, sex, diagnosis), rather than being removed.

In [ ]:
from fedbiomed.common.constants import PreprocType

# Caveat: pre-processing uses the whole dataset, even if training plan is filtering
# only some columns for training. So column numbers reference the *full* dataset
fedcombat_args = {
    # Can use numeric values
    #'covariates': [1,2],
    #'phenotypes': [3,4,5,6,7,8,9,10,11,12,13,14,15],

    # Can use label values
    'covariates': ['SEX','AGE','PTEDUCAT'],
    'phenotypes': [ 'CDRSB.bl','ADAS11.bl' ],
    #'phenotypes': [
    #    'CDRSB.bl','ADAS11.bl','MMSE.bl','RAVLT.immediate.bl','RAVLT.learning.bl','RAVLT.forgetting.bl',
    #    'FAQ.bl','WholeBrain.bl','Ventricles.bl','Hippocampus.bl','MidTemp.bl','Entorhinal.bl',
    #],

    # Can NOT mix numeric and label in same variable
    #'covariates': [1,'AGE','PTEDUCAT'],
    #'phenotypes': ['CDRSB.bl','ADAS11.bl',6,7],

    # Can NOT mix numeric and label even in the other variable
    #'covariates': ['SEX','AGE','PTEDUCAT'],
    #'phenotypes': [4,5,6,7],

    # Optionally save harmonized data in standardized scale
    # Note: only phenotypes are standardized
    # (default: False, final data in original scale)
    #'standardize_result': False
}

# associate a pre-processing with the experiment
# by default, when creating experiment, no pre-processing is configured
exp.set_preprocessing(PreprocType.FEDCOMBAT, fedcombat_args)

In [ ]:
# Optionally save initial federated dataset, for demonstration purpose of this example.
# Probably not needed in real life scenario.
import copy
initial_dataset = copy.deepcopy(exp.training_data().data())

### Harmonize and train model

Once a pre-processing is configured for the experiment, federated dataset is automatically harmonized before training if needed.

In [ ]:
# View initial dataset
exp.training_data().data()

In [ ]:

# first round: needs pre-preprocessing
exp.run_once(increase=True)

In [ ]:
# Harmonized dataset is saved on the nodes, and federated dataset is updated on the researcher
# Harmonized dataset is not searchable by tags, as it is mostly meant to be used within
# the context of the current experiment.

# View harmonized dataset
exp.training_data().data()

In [ ]:
# same context: does not need pre-processing again ...
exp.run_once(increase=True)

In [ ]:
# ... yet preprocessing is associated with the experiment
# exp.preprocessing

This is basically how pre-processing is used for an experiment: setup a preprocessing for the experiment, then everything is automatic !

### Advanced pre-processing handling

This part demonstrates how federated pre-processing behaves in more advanced scenarios.

#### Understand and control pre-processing execution

Pre-processing for an experiment needs to be executed once to harmonized a federated dataset before training.

Then it does not need to be re-executed if training more rounds, unless conditions changed: 
- using a different set of nodes
- using a different federated dataset
- using different pre-processing arguments

Examples below are given to demonstrate how pre-processing and training workflows interact. Yet, they are not a real scenario: in most cases it does not make sense to reapply harmonization several times to same dataset.

In [ ]:
# Optionally reuse initial dataset instead of harmonized dataset
# Can (should ?) be done after each harmonization
#
# When dataset changes, then pre-processing needs to be re-executed

exp.set_training_data(initial_dataset)

In [ ]:
exp.run_once(increase=True)

In [ ]:
# set again preprocessing: needs pre-processing - even with same conditions
exp.set_preprocessing(PreprocType.FEDCOMBAT, fedcombat_args)

In [ ]:
exp.run_once(increase=True)

In [ ]:
import copy
fedcombat_args2 = copy.deepcopy(fedcombat_args)
fedcombat_args2.update(
    {
        'rounds': 5,  # non-default number of training rounds for the harmonization model
        'training_args': {
            'loader_args': {'batch_size': 8},
            'log_interval': 10,  # reasonably verbose harmonization logging
        }
    }
)

In [ ]:
# set again preprocessing with different args: needs pre-processing
exp.set_preprocessing(PreprocType.FEDCOMBAT, fedcombat_args2)

In [ ]:
# pre-processing can also be triggered without training ...
exp.preprocessing.execute()

In [ ]:
# ... still it is run only if needed
exp.preprocessing.execute()

In [ ]:
# ... unless we force it
exp.preprocessing.execute(force=True)

In [ ]:
# So now pre-processing is not needed, as it was forced (just before) with same conditions
exp.run_once(increase=True)

In [ ]:
# Optionally test node filter: needs pre-processing if restricting used nodes
#exp.set_nodes(['replace_with_some_experiment_NODE_ID'])
#exp.run_once(increase=True)

In [ ]:
# Optionally remove node filter and reuse a dataset  with all availables nodes: needs pre-processing again
#exp.set_nodes(None)
#exp.run_once(increase=True)

In [ ]:
# no pre-processing - can also give PreprocType.NONE or False
exp.set_preprocessing(None)
exp.run_once(increase=True)

In [ ]:
exp.preprocessing

In [ ]:
# set again preprocessing: needs pre-processing
exp.set_preprocessing(PreprocType.FEDCOMBAT, fedcombat_args)
exp.run_once(increase=True)

In [ ]:
# check full pre-processing config at time of saving breakpoint
#exp.preprocessing.__dict__

#### Load breakpoint

In [ ]:
del exp

In [ ]:
loaded_exp = Experiment.load_breakpoint()

In [ ]:
# check pre-processing config after loading breakpoint : same configuration as the one saved
#loaded_exp.preprocessing.__dict__

In [ ]:
# Tensorboard needs to be manually set again after loading a breakpoint
#loaded_exp.set_tensorboard(True)

In [ ]:
loaded_exp.run_once(increase=True)